# 🇸🇩 التدريب العصبي التلقائي للهجة السودانية (One-Click Automated Fine-Tuning)
### تطوير أوزان نموذج XTTS-v2 ليتكلم باللكنة ومخارج الحروف السودانية الأصيلة
---
**تم التحديث لإلغاء الواجهة اليدوية المعقدة — التدريب والحفظ إلى Google Drive يتم الآن آلياً بالكامل:**
1. `▶` **الخلية 1**: تثبيت حزمة التدريب المتوافقة وربط Google Drive.
2. `▶` **الخلية 2**: تحميل البيانات المراجَعة (`_clean`) للتدريب والتقييم.
3. `▶` **الخلية 3**: تشغيل التدريب المباشر (Auto-Train) مع نقاط استئناف في Drive، وحفظ النموذج تلقائياً في Google Drive.
4. `▶` **الخلية 4**: تجربة وسماع الصوت السوداني المدرب!

## 1️⃣ الخلية الأولى: تثبيت حزمة التدريب وربط Google Drive
*(تستغرق دقيقة واحدة — انتظر ظهور علامة الصح الخضراء `✓`)*

In [11]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'

!pip install -q coqui-tts pandas soundfile

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

print('\n✅ تم تثبيت المحرك وربط Google Drive بنجاح!')


✅ تم تثبيت المحرك وربط Google Drive بنجاح!


## 2️⃣ الخلية الثانية: تجهيز وتقسيم البيانات للتدريب والتقييم (Auto-Split)
*(تستخدم `metadata_train_clean.csv` و `metadata_eval_clean.csv` التي راجعتها بنفسك إن وُجدت، وإلا تأخذ عينة من التفريغ الخام)*

In [ ]:
import pandas as pd
import os

DATASET_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuning_dataset'
CLEAN_TRAIN = os.path.join(DATASET_DIR, 'metadata_train_clean.csv')
CLEAN_EVAL = os.path.join(DATASET_DIR, 'metadata_eval_clean.csv')

if os.path.exists(CLEAN_TRAIN) and os.path.exists(CLEAN_EVAL):
    # ✅ المسار الموصى به: بيانات راجعتها بنفسك (data_cleaning/ → review_server.js → export_training_set.py)
    USING_CLEAN = True
    TRAIN_CSV, EVAL_CSV = CLEAN_TRAIN, CLEAN_EVAL
    train_df = pd.read_csv(TRAIN_CSV, sep='|')
    eval_df = pd.read_csv(EVAL_CSV, sep='|')
    print(f'✅ استخدام البيانات المنظّفة والمراجَعة: تدريب {len(train_df):,} | اختبار {len(eval_df):,}')
else:
    # ⚠️ الطريقة القديمة: عينة عشوائية من تفريغ Whisper الخام دون مراجعة (جودة النص فيها ضعيفة)
    USING_CLEAN = False
    METADATA_FILE = os.path.join(DATASET_DIR, 'metadata.csv')
    TRAIN_CSV = os.path.join(DATASET_DIR, 'metadata_train.csv')
    EVAL_CSV = os.path.join(DATASET_DIR, 'metadata_eval.csv')

    df = pd.read_csv(METADATA_FILE, sep='|', on_bad_lines='skip').dropna()
    df = df[df['text'].str.len() >= 8]

    MAX_SAMPLES = 8000
    if MAX_SAMPLES and len(df) > MAX_SAMPLES:
        df = df.sample(n=MAX_SAMPLES, random_state=42).reset_index(drop=True)
    else:
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    split_idx = int(len(df) * 0.9)
    train_df = df.iloc[:split_idx]
    eval_df = df.iloc[split_idx:]
    train_df.to_csv(TRAIN_CSV, sep='|', index=False)
    eval_df.to_csv(EVAL_CSV, sep='|', index=False)
    print('⚠️ لم أجد metadata_train_clean.csv — سأدرّب على عينة عشوائية من التفريغ الخام (غير موصى به).')
    print(f'عينات التدريب: {len(train_df):,} | الاختبار: {len(eval_df):,}')

display(train_df.head(5))


## 3️⃣ الخلية الثالثة: إطلاق التدريب التلقائي وحفظ النموذج في Drive مباشرة
*(ستبدأ العملية آلياً وتظهر لك خطوات التدريب وتناقص نسبة الخطأ، وتحفظ النموذج النهائي مباشرة داخل مجلد المشروع على Google Drive)*

**إذا انقطعت الجلسة:** كل 3 دورات تُحفظ نقطة استئناف في `training_resume` على Drive. أعد الاتصال وشغّل الخلايا 1 و2 و3 من جديد، فيكمل التدريب من آخر نقطة بدل البدء من الصفر.

In [ ]:
import os, glob, shutil
import torch
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.xtts_config import XttsAudioConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
from TTS.utils.manage import ModelManager

os.environ["COQUI_TOS_AGREED"] = "1"

DATASET_DIR = "/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuning_dataset"
TRAIN_CSV = globals().get("TRAIN_CSV") or os.path.join(DATASET_DIR, "metadata_train.csv")
EVAL_CSV = globals().get("EVAL_CSV") or os.path.join(DATASET_DIR, "metadata_eval.csv")
# مجموعة صغيرة ونظيفة تحتاج دورات أكثر وتحديثات أكثر (batch فعلي أصغر) من مجموعة كبيرة خام
USING_CLEAN = globals().get("USING_CLEAN", False)
EPOCHS = 20 if USING_CLEAN else 4
GRAD_ACCUM = 4 if USING_CLEAN else 16
OUT_DRIVE_DIR = "/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuned_model"
BASE_DIR = "/content/xtts_base_v2"
RUN_OUT = "/content/xtts_ft_run"
os.makedirs(OUT_DRIVE_DIR, exist_ok=True)
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(RUN_OUT, exist_ok=True)

# 💾 نقطة استئناف في Drive: إذا انقطعت جلسة Colab، أعد تشغيل الخلايا كلها فيكمل التدريب من آخر نقطة محفوظة
RESUME = True            # اجعلها False لتبدأ تدريبًا جديدًا من الصفر وتتجاهل أي نقطة محفوظة
SAVE_EVERY_EPOCHS = 3    # كل كم دورة تُحفظ نقطة استئناف (~2GB) في Drive
RESUME_DIR = "/content/drive/MyDrive/Sudanese_Voice_Project/training_resume"
RESUME_CKPT = os.path.join(RESUME_DIR, "resume.pth")
RESUME_INFO = os.path.join(RESUME_DIR, "resume.json")
os.makedirs(RESUME_DIR, exist_ok=True)

# 1️⃣ تنزيل ملفات XTTS-v2 الأساسية مباشرة من HuggingFace
HF = "https://huggingface.co/coqui/XTTS-v2/resolve/main"
links = [
    f"{HF}/dvae.pth",
    f"{HF}/mel_stats.pth",
    f"{HF}/vocab.json",
    f"{HF}/model.pth",
    f"{HF}/config.json",
    f"{HF}/speakers_xtts.pth",
]
needed = [u for u in links if not os.path.isfile(os.path.join(BASE_DIR, os.path.basename(u)))]
if needed:
    print("⏳ جاري تنزيل أوزان XTTS-v2 الأساسية من HuggingFace...")
    ModelManager._download_model_files(needed, BASE_DIR, progress_bar=True)

DVAE_CHECKPOINT = os.path.join(BASE_DIR, "dvae.pth")
MEL_NORM_FILE   = os.path.join(BASE_DIR, "mel_stats.pth")
TOKENIZER_FILE  = os.path.join(BASE_DIR, "vocab.json")
XTTS_CHECKPOINT = os.path.join(BASE_DIR, "model.pth")
XTTS_CONFIG     = os.path.join(BASE_DIR, "config.json")
XTTS_SPEAKERS   = os.path.join(BASE_DIR, "speakers_xtts.pth")

# 2️⃣ إعداد الداتاسيت ومعاملات التدريب المتوافقة تماماً مع coqui-tts 0.28+
config_dataset = BaseDatasetConfig(
    formatter="coqui",
    dataset_name="sudanese_dialect",
    path=DATASET_DIR,
    meta_file_train=TRAIN_CSV,
    meta_file_val=EVAL_CSV,
    language="ar",
)

model_args = GPTArgs(
    max_conditioning_length=132300,
    min_conditioning_length=66150,
    debug_loading_failures=False,
    max_wav_length=255995,
    max_text_length=200,
    mel_norm_file=MEL_NORM_FILE,
    dvae_checkpoint=DVAE_CHECKPOINT,
    xtts_checkpoint=XTTS_CHECKPOINT,
    tokenizer_file=TOKENIZER_FILE,
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)

audio_config = XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000)

config = GPTTrainerConfig(
    epochs=EPOCHS,
    output_path=RUN_OUT,
    model_args=model_args,
    run_name="GPT_XTTS_Sudanese_FT",
    project_name="Sudanese_Voice_AI",
    run_description="Sudanese Dialect XTTS-v2 Fine-Tuning",
    datasets=[config_dataset],
    dashboard_logger="tensorboard",
    audio=audio_config,
    batch_size=2,
    batch_group_size=48,
    eval_batch_size=2,
    num_loader_workers=2,
    eval_split_max_size=256,
    print_step=50,
    plot_step=250,
    log_model_step=500,
    save_step=1000,
    save_n_checkpoints=1,
    save_checkpoints=True,
    print_eval=False,
    optimizer="AdamW",
    optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=5e-06,
    lr_scheduler="MultiStepLR",
    lr_scheduler_params={"milestones": [50000 * 18, 150000 * 18, 300000 * 18], "gamma": 0.5, "last_epoch": -1},
    test_sentences=[],
)

# 3️⃣ بناء النموذج وتحميل العينات بالواجهة الرسمية الحديثة (coqui-tts 0.28+)
model = GPTTrainer(config)

# ♻️ الاستئناف: نقطة الاستئناف لا تُستخدم إلا إن كانت لنفس ملفات البيانات ونفس إعدادات التدريب
import json, hashlib

def _fingerprint():
    h = hashlib.md5()
    for p in (TRAIN_CSV, EVAL_CSV):
        with open(p, "rb") as f:
            h.update(f.read())
    h.update(f"{EPOCHS}|{GRAD_ACCUM}".encode())
    return h.hexdigest()

FINGERPRINT = _fingerprint()
START_EPOCH = 0
if RESUME and os.path.isfile(RESUME_CKPT) and os.path.isfile(RESUME_INFO):
    with open(RESUME_INFO) as f:
        info = json.load(f)
    if info.get("fingerprint") == FINGERPRINT:
        state = torch.load(RESUME_CKPT, map_location="cpu", weights_only=False)
        done = int(state.get("epochs_done", 0))
        if 0 < done < EPOCHS:
            model.load_state_dict(state["model"])
            START_EPOCH = done
            print(f"♻️ وجدت نقطة استئناف بعد {done} من {EPOCHS} دورة — سأكمل من الدورة {done + 1}.")
        del state
    else:
        print("ℹ️ نقطة الاستئناف الموجودة تخص بيانات أو إعدادات أخرى — سأبدأ من الصفر.")
config.epochs = EPOCHS - START_EPOCH


def save_resume(trainer):
    """تُستدعى بعد كل دورة: تحفظ أوزان النموذج (دون حالة المحسِّن) في Drive كل SAVE_EVERY_EPOCHS دورات."""
    done = START_EPOCH + trainer.epochs_done + 1
    if done % SAVE_EVERY_EPOCHS != 0 or done >= EPOCHS:
        return
    tmp_local = "/content/resume_tmp.pth"
    torch.save({"model": trainer.model.state_dict(), "epochs_done": done}, tmp_local)
    shutil.copyfile(tmp_local, RESUME_CKPT + ".tmp")
    os.replace(RESUME_CKPT + ".tmp", RESUME_CKPT)
    os.remove(tmp_local)
    with open(RESUME_INFO, "w") as f:
        json.dump({"epochs_total": EPOCHS, "fingerprint": FINGERPRINT}, f)
    print(f"💾 حُفظت نقطة استئناف في Drive بعد الدورة {done} من {EPOCHS}")
train_samples, eval_samples = load_tts_samples(
    config,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)
print(f"🚀 جاهز! تم تحميل {len(train_samples):,} عينة تدريب و {len(eval_samples):,} عينة اختبار — "
      f"تدريب الدورات {START_EPOCH + 1} إلى {EPOCHS} على كرت الشاشة...")

trainer = Trainer(
    TrainerArgs(
        restore_path=None,
        skip_train_epoch=False,
        start_with_eval=False,
        grad_accum_steps=GRAD_ACCUM,
    ),
    config,
    output_path=RUN_OUT,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
    callbacks={"on_epoch_end": save_resume},
)
trainer.fit()

# 4️⃣ نسخ أفضل نموذج مدرب مع ملفات التشغيل إلى Google Drive
candidates = glob.glob(os.path.join(trainer.output_path, "**", "best_model*.pth"), recursive=True)
if not candidates:
    candidates = glob.glob(os.path.join(trainer.output_path, "**", "checkpoint_*.pth"), recursive=True)
if candidates:
    best_ckpt = max(candidates, key=os.path.getctime)
    out_model = os.path.join(OUT_DRIVE_DIR, "model.pth")
    try:
        # checkpoint التدريب الخام ~5.6GB (فيه optimizer وأوزان dvae). نحذفهما كما تفعل وصفة Coqui الرسمية
        # فيصبح الملف بحجم النموذج الأساسي تقريبًا (~1.9GB) ويتحمّل أسرع ويستهلك ذاكرة أقل.
        ckpt = torch.load(best_ckpt, map_location="cpu", weights_only=False)
        ckpt.pop("optimizer", None)
        for k in list(ckpt["model"].keys()):
            if "dvae" in k:
                del ckpt["model"][k]
        torch.save(ckpt, out_model)
        print(f"✅ تم حفظ أوزان سودانية مخففة ({os.path.basename(best_ckpt)}) — {os.path.getsize(out_model)/1e9:.2f}GB")
    except Exception as e:
        print(f"⚠️ تعذر تخفيف الـ checkpoint ({e}); سأنسخه كما هو.")
        shutil.copy2(best_ckpt, out_model)
    # اكتمل التدريب وحُفظ النموذج: نقطة الاستئناف لم تعد لازمة (تذهب إلى سلة مهملات Drive)
    for p in (RESUME_CKPT, RESUME_INFO):
        if os.path.exists(p):
            os.remove(p)
shutil.copy2(XTTS_CONFIG, os.path.join(OUT_DRIVE_DIR, "config.json"))
shutil.copy2(TOKENIZER_FILE, os.path.join(OUT_DRIVE_DIR, "vocab.json"))
shutil.copy2(XTTS_SPEAKERS, os.path.join(OUT_DRIVE_DIR, "speakers_xtts.pth"))
print("\n🎉 اكتمل التدريب العصبي وحُفظ النموذج السوداني المطور بنجاح في Google Drive!")


## 4️⃣ الخلية الرابعة: تجربة وسماع الصوت السوداني المدرب فوراً
*(يقرأ النموذج المدرب من مجلد `fine_tuned_model` في Drive ويولد الجملة بالنبرة السودانية الأصيلة)*

In [ ]:
import torch
from TTS.api import TTS
from IPython.display import Audio, display
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
FINE_TUNED_MODEL_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuned_model'
REF_WAV = '/content/drive/MyDrive/Sudanese_Voice_Project/corpus/references/clean_podcast_8s.wav'

if os.path.exists(os.path.join(FINE_TUNED_MODEL_DIR, 'model.pth')):
    print('🎉 جاري تحميل النموذج السوداني المدرب حديثاً...')
    xtts_tuned = TTS(model_path=FINE_TUNED_MODEL_DIR, config_path=os.path.join(FINE_TUNED_MODEL_DIR, 'config.json')).to(device)
else:
    print('⚠️ لم يكتمل حفظ model.pth بعد، جاري استخدام النموذج الأساسي.')
    xtts_tuned = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

test_phrase = 'يا زول مرحب بيك في استوديو الصوت السوداني، دحين كدا النطق بقى سمح بالحيل وظابط تمام.'
out_audio = '/content/drive/MyDrive/Sudanese_Voice_Project/cloned_sudanese.wav'

xtts_tuned.tts_to_file(
    text=test_phrase,
    speaker_wav=REF_WAV,
    language='ar',
    file_path=out_audio,
    temperature=0.6,
    repetition_penalty=2.0
)

print('✅ تم التوليد والحفظ في Drive بنجاح!')
display(Audio(out_audio, autoplay=True))


🎉 جاري تحميل النموذج السوداني المدرب حديثاً...
